# Dogs and training outcomes

This chapter describes which dogs are included in the study and how their training outcomes are used to form comparison groups. The cohort includes Labrador Retrievers and German Shorthaired Pointers, with breed, sex, and outcomes taken from the January 2024 records.

A dog is included only when its training record can be linked uniquely to a genetic sample. Records with missing or ambiguous links are excluded. We use the January records on the investigator's assumption that historical corrections were completed; this assumption does not independently establish every dog's identity.


## Defining the comparison groups

Dogs recorded as Eliminated for a Behavioral reason are cases in both comparisons. We treat all behavioral elimination reasons together.

In **Scheme A**, the controls are dogs recorded as Validated or Certified with no elimination reason. Certified is treated as equivalent to Validated for this purpose. This is the primary comparison for testing the earlier Labrador findings.

In **Scheme B**, the controls also include dogs recorded as Eliminated for Medical, Environmental, or Inconsistency reasons. This broader comparison tests whether the findings depend on which dogs are used as controls. It does not establish that these other outcomes are unrelated to behavior.

Dogs with unresolved outcomes or unknown elimination reasons are excluded. The original records remain unchanged.

| Comparison | Breed | Controls | Behavioral cases | Total |
| --- | --- | ---: | ---: | ---: |
| A: Validated or Certified controls | German Shorthaired Pointer | 96 | 9 | 105 |
| A: Validated or Certified controls | Labrador Retriever | 39 | 7 | 46 |
| B: Broader control group | German Shorthaired Pointer | 107 | 9 | 116 |
| B: Broader control group | Labrador Retriever | 41 | 7 | 48 |

The broader comparison adds 11 German Shorthaired Pointer controls and two Labrador Retriever controls. It includes all dogs in the main comparison and uses the same cases. The two comparisons therefore overlap and do not represent independent groups of dogs.


## Checking the prepared data

The training records and genetic samples have been linked, and the comparison groups have been prepared. We checked that each included dog has a unique study identifier, that the case groups match across comparisons, and that the genetic measurements copied into the prepared datasets match the original measurements.

The genetic data contain 229,991 measured markers, or locations in the genome. No markers have been removed at this stage. The counts above describe dogs eligible from their records; genetic quality checks may change the numbers available for analysis.

Study identifiers are used in place of original dog identifiers, and the same dog keeps the same identifier across comparisons. Individual outcomes, genetic records, and the link to original identifiers remain private.

The next chapter reports the quality checks applied to these genetic data before association testing.


## Load the prepared study groups

This chapter starts from the certified-inclusive cohorts already prepared from the January records. Rebuilding those cohorts from raw company records still uses privately archived preparation scripts; this edition does not imply that raw reconciliation is yet reproduced by a public script. The cells below independently read the selected genotype files, verify their recorded versions, and count the case and control groups.

In [ ]:
from pathlib import Path
import sys

study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
from notebook_support import read_json, study_path, verify_datasets, run_script

In [ ]:
inputs = read_json(study_root / "data/input/manifest.private.json")
verify_datasets(study_root, inputs)
print("Prepared cohort version:", inputs["version"])
print("Verified genotype datasets:", len(inputs["datasets"]))

### Count dogs from their genotype sample files

PLINK's sample file records phenotype code 1 for a control and 2 for a behavioral case. This function counts those codes without printing individual identifiers. It also checks that the dog identifiers are unique within each dataset.

In [ ]:
from collections import Counter

def cohort_counts(prefix):
    rows = [line.split() for line in prefix.with_suffix(".fam").read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len({row[1] for row in rows}) == len(rows)
    phenotypes = Counter(row[5] for row in rows)
    assert set(phenotypes) <= {"1", "2"}
    return {"dogs": len(rows), "controls": phenotypes["1"], "cases": phenotypes["2"]}

for dataset in inputs["datasets"]:
    prefix = next(study_path(study_root, p) for p in dataset["files"] if p.endswith(".fam")).with_suffix("")
    counts = cohort_counts(prefix)
    assert all(counts[key] == dataset[key] for key in counts)
    print(dataset["scenario"], dataset["breed"], counts)